In [1]:
import time
from decimal import Decimal
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
import psycopg2
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel, AutoModelForSequenceClassification
import torch

# Optional local model fallback
try:
    from sentence_transformers import SentenceTransformer
    local_model = SentenceTransformer('all-MiniLM-L6-v2')
except ImportError:
    local_model = None

load_dotenv()

True

In [3]:
# Define some constants
USE_OPENAI = False

# Define your index's expected dimension
INDEX_DIM = 384  # Change if your Pinecone index was created with a different dimension

In [4]:
PG_QUERY = """
WITH subway_flat AS (
    SELECT
        ls.listing_id AS listing_pk,              -- FK to listings.id
        l.listing_id AS listing_id_str,           -- StreetEasy string id
        LOWER(s.line) AS line,
        LOWER(unnest(coalesce(s.routes, ARRAY[]::text[]))) AS route,
        ls.distance
    FROM listing_subway ls
    JOIN listings l
        ON l.id = ls.listing_id                   -- match FK to PK
    JOIN subway_stations s
        ON s.id = ls.subway_id                    -- match FK to PK
),
subway_grouped AS (
    SELECT
        listing_id_str,
        line,
        route,
        MIN(distance) AS min_distance
    FROM subway_flat
    GROUP BY listing_id_str, line, route
)
SELECT
    l.listing_id,       -- StreetEasy string id
    l.description,
    l.price,
    l.bedrooms,
    l.bathrooms,
    l.sqft,
    l.neighborhood,
    l.borough,
    l.zipcode,
    l.built_in,
    b.address AS building_address,
    b.latitude AS building_lat,
    b.longitude AS building_lon,
    b.built_in AS building_built_in,
    array_to_string(l.amenities, ', ') AS amenity_list,

    STRING_AGG(
        sg.line || ' (' || sg.route || ') Train (' || sg.min_distance || ' miles)',
        '; ' ORDER BY sg.min_distance ASC
    ) AS subway_info,

    ARRAY_AGG(DISTINCT sg.line) AS subway_lines,
    ARRAY_AGG(DISTINCT sg.route) AS subway_routes,
    ARRAY_AGG(sg.min_distance) AS subway_distances

FROM listings l
LEFT JOIN buildings b 
    ON b.id = l.building_id
LEFT JOIN subway_grouped sg 
    ON sg.listing_id_str = l.listing_id   -- StreetEasy string match
WHERE l.status = 'open'
GROUP BY
    l.listing_id, l.description, l.price, l.bedrooms,
    l.bathrooms, l.sqft, l.neighborhood, l.borough,
    l.zipcode, l.built_in, b.address, b.latitude,
    b.longitude, b.built_in, l.amenities;
"""

In [5]:
def running_in_docker() -> bool:
    """Detect if we're running inside a Docker container."""
    try:
        with open('/proc/1/cgroup', 'rt') as f:
            content = f.read()
            return 'docker' in content or 'containerd' in content
    except FileNotFoundError:
        return False

def get_connection():
    """Connect to Postgres correctly depending on environment."""
    if running_in_docker():
        resolved_host = "db"  # inside Docker network
    else:
        resolved_host = "localhost"  # force localhost when outside

    print(f"[DEBUG] Connecting to Postgres at host: {resolved_host}")

    return psycopg2.connect(
        dbname=os.getenv("DATABASE_NAME", "rentiq_db"),
        user=os.getenv("DATABASE_USER", "rentiq_user"),
        password=os.getenv("DATABASE_PASSWORD", "RentIQ2025!"),
        host=resolved_host,
        port=os.getenv("DATABASE_PORT", "5432")
    )

def fetch_listings():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute(PG_QUERY)
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return rows

def build_text(row):
    (
        listing_id, description, price, bedrooms, bathrooms, sqft,
        neighborhood, borough, zipcode, built_in, building_address,
        building_lat, building_lon, building_built_in, amenity_list,
        subway_info, subway_lines, subway_routes, subway_distances
    ) = row

    amenity_list = amenity_list or "No amenities listed"
    subway_info = subway_info or "No nearby subway stations"

    return f"""Listing ID: {listing_id}
{description}
Located at {building_address} in {neighborhood}, {borough} ({zipcode}).
Price: ${price}, {bedrooms} bedrooms, {bathrooms} bathrooms, {sqft} sqft.
Amenities: {amenity_list}.
Nearby subway stations: {subway_info}.
Built in {building_built_in}.
"""

In [6]:
listings = fetch_listings()

[DEBUG] Connecting to Postgres at host: localhost


In [7]:
print(f"Fetched {len(listings)} listings from the database.")

Fetched 6061 listings from the database.


In [9]:
data_rows = []
for row in listings:
    data_rows.append(build_text(row))
    
print(data_rows[0])

Listing ID: 3027856
Welcome to The Atelier Condominium, Manhattan’s most coveted luxury high-rise, a true architectural icon and pinnacle of sophisticated living. Designed by award-winning architect Costas Kondylis, The Atelier redefines upscale city living. This is not just a residence—it’s a lifestyle reserved for the discerning.

Unlike standard rental buildings, condominium living offers a heightened level of privacy, security, exclusivity, and attention to detail. At The Atelier, every residence is impeccably maintained to ownership standards, resulting in a quieter, more refined community where pride of place and quality craftsmanship are paramount. This is luxury reimagined—where discerning renters benefit from elevated services, enhanced amenities, and a sense of permanence not found in traditional rentals.

Elevate your expectations and experience why The Atelier stands apart as a preferred home to celebrities and tastemakers alike.

Our Atelier Condo Luxury Rental Office is o

## Set up Pinecone Vectorstore

### Initialize Pinecone and Create the Hybrid Index

In [10]:
# Initialize Pinecone and OpenAI clients
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Connect to Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [11]:
index_name = "listings-index-hybrid"  # name for our hybrid index

# === Delete old index if exists ===
if index_name in [idx.name for idx in pc.list_indexes()]:
    print(f"[INFO] Deleting existing index '{index_name}'")
    pc.delete_index(index_name)

# Create the index if it doesn't exist yet
if index_name not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,  # dense vector dimension (MiniLM-L6-v2)
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
print("[INFO] Hybrid Pinecone index ready.")

[INFO] Deleting existing index 'listings-index-hybrid'
[INFO] Hybrid Pinecone index ready.


### Load Dense & Sparse Models

In [ ]:
# Dense model
dense_model = SentenceTransformer('all-MiniLM-L6-v2')

# Sparse (SPLADE) model
splade_model_name = "naver/splade-cocondenser-ensembledistil"
splade_tokenizer = AutoTokenizer.from_pretrained(splade_model_name)
splade_model = AutoModelForMaskedLM.from_pretrained(splade_model_name)

# Load BGE Reranker model
rerank_model_name = "BAAI/bge-reranker-v2-m3"
rerank_tokenizer = AutoTokenizer.from_pretrained(rerank_model_name)
rerank_model = AutoModelForSequenceClassification.from_pretrained(rerank_model_name)

In [ ]:
def splade_encode(text: str):
    """Generate sparse vector for Pinecone from SPLADE."""
    inputs = splade_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = splade_model(**inputs).logits
    weights = torch.log1p(torch.relu(logits)).max(dim=1).values.squeeze()
    indices = torch.nonzero(weights, as_tuple=True)[0].tolist()
    values = weights[indices].tolist()
    return {"indices": indices, "values": values}

In [ ]:
# ------------ SANITIZE METADATA ----------------
def sanitize_metadata(metadata: dict) -> dict:
    clean = {}
    for k, v in metadata.items():
        if v is None:
            continue
        elif isinstance(v, (int, float, bool, str, list, dict)):
            clean[k] = v
        elif isinstance(v, Decimal):
            clean[k] = float(v)
        else:
            clean[k] = str(v)  # fallback
    return clean


In [ ]:
BATCH_SIZE = 100
rows = fetch_listings()
print(f"[INFO] Loaded {len(rows)} listings from Postgres")

In [ ]:
batch = []
for i, row in enumerate(rows, start=1):
    text = build_text(row)

    # Dense
    dense_vec = dense_model.encode(text).tolist()

    # Sparse
    sparse_vec = splade_encode(text)

    (
        listing_id,                      # StreetEasy string
        description,
        price,
        bedrooms,
        bathrooms,
        sqft,
        neighborhood,
        borough,
        zipcode,
        built_in,
        building_address,
        building_lat,
        building_lon,
        building_built_in,
        amenity_list,
        subway_info,                      # human-readable string
        subway_lines,                     # array aligned with routes/distances
        subway_routes,
        subway_distances
    ) = row
    
    # --- Safety: Default fallbacks ---
    amenity_list = amenity_list or "No amenities listed"
    subway_info = subway_info or "No nearby subway stations"
    borough = borough.lower() if borough else "Not available"
    neighborhood = neighborhood.lower() if neighborhood else "Not available"
    
    # --- Subway distances map (key = line_route) ---
    subway_dist_map = {}
    if subway_lines and subway_routes and subway_distances:
        for line, route, dist in zip(subway_lines, subway_routes, subway_distances):
            if line and route and dist is not None:
                key = f"{line.lower()}_{route.lower()}"
                subway_dist_map[key] = float(dist)
                
    # Convert dict to list of "key:distance" strings
    subway_dist_list = [f"{key}:{dist}" for key, dist in subway_dist_map.items()]

    # --- Metadata ---
    metadata = sanitize_metadata({
        "listing_id": listing_id,
        "price": price,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "sqft": sqft,
        "borough": borough,
        "neighborhood": neighborhood,
        "zipcode": zipcode,
        "building_address": building_address,
        "amenities": [a.strip().lower() for a in amenity_list.split(",")] if amenity_list and "no amenities" not in amenity_list.lower() else [],
        "subway_info": subway_info,
        "subway_lines": [line.lower() for line in subway_lines if line],
        "subway_routes": [route.lower() for route in subway_routes if route],
        #"subway_distances": subway_dist_list, ## TODO Add a better representation for filters
        "description": description
    })

    # --- Add to batch ---
    batch.append({
        "id": listing_id,   # use StreetEasy id as Pinecone vector id
        "values": dense_vec,
        "sparse_values": sparse_vec,
        "metadata": metadata
    })

    if len(batch) >= BATCH_SIZE:
        index.upsert(batch)
        print(f"[INFO] Upserted batch of {len(batch)} listings")
        batch.clear()

# --- Final flush ---
if batch:
    index.upsert(batch)
    print(f"[INFO] Final batch upserted ({len(batch)} listings)")

print("[✅] Hybrid Pinecone ingestion complete.")

In [ ]:
def hybrid_search_raw(query_text: str, alpha: float = 0.5, top_k: int = 20, filters: dict = None):
    dense_q = dense_model.encode(query_text).tolist()
    sparse_q = splade_encode(query_text)

    beta = 1 - alpha
    weighted_dense = [v * alpha for v in dense_q]
    weighted_sparse = {
        "indices": sparse_q["indices"],
        "values": [v * beta for v in sparse_q["values"]]
    }

    query_params = {
        "vector": weighted_dense,
        "sparse_vector": weighted_sparse,
        "top_k": top_k,
        "include_metadata": True,
        "include_values" : True
    }
    if filters:
        query_params["filter"] = filters

    results = index.query(**query_params)
    return results.matches

In [ ]:
def deduplicate_matches(matches):
    seen_ids = set()
    unique_matches = []
    for match in matches:
        listing_id = match.metadata.get("listing_id")
        if listing_id not in seen_ids:
            seen_ids.add(listing_id)
            unique_matches.append(match)
    return unique_matches

In [ ]:
def rerank_results(query: str, matches):
    """Rerank matches using BGE Reranker."""
    docs = [m.metadata.get("description", "") for m in matches]
    pairs = [(query, doc) for doc in docs]

    inputs = rerank_tokenizer(pairs, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        scores = rerank_model(**inputs).logits.squeeze(1)  # Higher = more relevant

    # Attach scores and sort
    scored_matches = [
        (match, score.item()) for match, score in zip(matches, scores)
    ]
    scored_matches.sort(key=lambda x: x[1], reverse=True)

    # Return sorted matches
    return scored_matches

In [ ]:
def hybrid_search(query_text: str, alpha: float = 0.5, top_k: int = 20, filters: dict = None, rerank: bool = True):
    # Step 1: Raw hybrid search
    matches = hybrid_search_raw(query_text, alpha=alpha, top_k=top_k, filters=filters)

    # Step 2: Deduplicate by listing_id
    matches = deduplicate_matches(matches)

    # Step 3: Optional rerank
    if rerank:
        matches_with_scores = rerank_results(query_text, matches)
        matches = [m for m, _ in matches_with_scores]

    # Final display
    for match in matches[:5]:  # show top 5 after rerank
        md = match.metadata
        print(f"[Score: {match.score:.4f}] ID: {md.get('listing_id')}, "
              f"${md.get('price')}, {md.get('bedrooms')}BR/{md.get('bathrooms')}BA, "
              f"{md.get('neighborhood')}, {md.get('borough')}")
        print(f"Amenities: {md.get('amenities')}")
        print("-----")

    return matches

In [ ]:
final_docs = hybrid_search(
    "pet-friendly 2 bedroom near F train in Manhattan under $2000",
    alpha=0.6,
    top_k=10,
    filters={"price": {"$lt": 4000}, "borough": {"$eq": "manhattan"}},
    rerank=True
)

## Write code for automating filter extraction process

In [ ]:
def build_filter_extraction_prompt(user_query: str) -> str:
    return f"""
You are an information extraction system for a real estate search engine.
Extract **only structured filters** from the given natural language rental search query for retrieval from a Pinecone vectorstore.

OUTPUT RULES:
- Format: Valid JSON object only — no comments, no explanations, no trailing commas.
- Only use fields present in the Pinecone metadata schema below:
    - listing_id (string)
    - price (number)
    - bedrooms (integer)
    - bathrooms (float)
    - sqft (integer)
    - borough (string, always lowercase)
    - neighborhood (string, always lowercase)
    - zipcode (string)
    - building_address (string)
    - amenities (array of lowercase strings)
    - subway_info (string)
    - subway_lines (array of lowercase strings)
    - subway_routes (array of lowercase strings)
    - description (string)
- All string values must be lowercase to match metadata.
- Arrays must use $in operator with a plain list of values, e.g. {{ "amenities": {{ "$in": ["elevator", "doorman"] }} }}
- Numeric comparisons: Use only $lt, $gt, $eq operators (single operator per field unless a range is given).
- Ignore descriptive text that doesn’t map to known metadata fields.
- Do not use fields not present in the metadata above.

EXAMPLES:

Query: "2 bedroom rental under $2000 in manhattan with elevator and doorman"
Output:
{{
    "price": {{"$lt": 2000}},
    "bedrooms": {{"$eq": 2}},
    "borough": {{"$eq": "manhattan"}},
    "amenities": {{"$in": ["elevator", "doorman"]}}
}}

Query: "No fee apartment near the f train under $3500 with 1.5 baths"
Output:
{{
    "subway_routes": {{"$in": ["f"]}},
    "price": {{"$lt": 3500}},
    "bathrooms": {{"$eq": 1.5}}
}}

Query: "Studio in manhattan near a train, pet friendly"
Output:
{{
    "borough": {{"$eq": "manhattan"}},
    "bedrooms": {{"$eq": 0}},
    "subway_routes": {{"$in": ["a"]}},
    "amenities": {{"$in": ["pet friendly"]}}
}}

USER QUERY:
{user_query}

Now output only the JSON filters.
"""

In [ ]:
import json
def process_user_query(user_query: str):
    prompt = build_filter_extraction_prompt(user_query)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    filters = json.loads(resp.choices[0].message.content)
    return filters

In [ ]:
user_query = "2 bed in manhattan under $2500, near A train"

In [ ]:
filters = process_user_query(user_query)
print("Extracted Filters:", filters)

In [ ]:
result = hybrid_search(
    user_query,
    alpha=0.5,
    top_k=10,
    filters={'price': {'$lt': 2500}, 'subway_routes': {'$in': ['a']}},
    rerank=True
)